In [1]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller
import os

DATA_PATH = r"..\..\Data\Main\modelling_dataset.csv"
OUTPUT_DIR = r"..\..\Dissertation_Results\EDA"
os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])

# Compute summary
non_zero = (df['sentiment'] != 0).sum()
total = len(df)

summary = {
    'Total observations': f"{total:,}",
    'Unique tickers': f"{df['ticker'].nunique()}",
    'Date range': f"{df['date'].min().date()} to {df['date'].max().date()}",
    'Target: positive (%)': f"{df['target'].mean()*100:.1f}%",
    'Target: negative (%)': f"{(1-df['target'].mean())*100:.1f}%",
    'Non-zero sentiment (%)': f"{non_zero/total*100:.1f}%",
    'Zero sentiment (%)': f"{(total-non_zero)/total*100:.1f}%",
    'Mean sentiment': f"{df['sentiment'].mean():.4f}",
    'Mean daily log return': f"{df['log_return'].mean():.4f}",
    'Mean 20-day volatility': f"{df['volatility_20d'].mean():.4f}",
}

summary_df = pd.DataFrame(list(summary.items()), columns=['Metric', 'Value'])
print(summary_df.to_string(index=False))

summary_df.to_csv(os.path.join(OUTPUT_DIR, "Table_4_1_Dataset_Summary.csv"), index=False)
print(f"\nSaved: Table_4_1_Dataset_Summary.csv")

                Metric                    Value
    Total observations                  589,886
        Unique tickers                      523
            Date range 2020-01-02 to 2025-12-31
  Target: positive (%)                    51.7%
  Target: negative (%)                    48.3%
Non-zero sentiment (%)                    75.5%
    Zero sentiment (%)                    24.5%
        Mean sentiment                  -0.0086
 Mean daily log return                   0.0004
Mean 20-day volatility                   0.0215

Saved: Table_4_1_Dataset_Summary.csv


In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import os

DATA_PATH = r"..\..\Data\Main\modelling_dataset.csv"
OUTPUT_DIR = r"..\..\Dissertation_Results\Tables"

df = pd.read_csv(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])
model_df = df.dropna(subset=['volatility_20d', 'next_day_return']).copy()
features = ['sentiment', 'volatility_20d', 'sent_x_vol']

full_data = model_df.dropna(subset=features)
X = sm.add_constant(full_data[features])
y = full_data['target']
result = sm.Logit(y, X).fit(disp=0)

full_period_table = pd.DataFrame({
    'Variable': ['Constant', 'Sentiment', 'Volatility_20d', 'Sentiment × Volatility'],
    'Coefficient': result.params.round(4).values,
    'Std Error': result.bse.round(4).values,
    'z-statistic': result.tvalues.round(3).values,
    'p-value': result.pvalues.round(4).values,
})

print(f"N = {len(full_data):,}\n")
print(full_period_table.to_string(index=False))

full_period_table.to_csv(os.path.join(OUTPUT_DIR, "Table_Full_Period_Regression.csv"), index=False)
print(f"\nSaved: Table_Full_Period_Regression.csv")

N = 578,903

              Variable  Coefficient  Std Error  z-statistic  p-value
              Constant       0.0686     0.0051       13.553   0.0000
             Sentiment      -0.0571     0.0163       -3.512   0.0004
        Volatility_20d       0.1003     0.2006        0.500   0.6170
Sentiment × Volatility       1.3595     0.6345        2.143   0.0321

Saved: Table_Full_Period_Regression.csv


In [3]:
import pandas as pd
import os

OUTPUT_DIR = r"..\..\Dissertation_Results\Prediction"
results = pd.read_csv(os.path.join(OUTPUT_DIR, "prediction_results_final_3x4.csv"))

# Clean feature set labels
label_map = {
    'A_sentiment_only': 'Sentiment only',
    'B_volatility_only': 'Volatility only',
    'C_sent_plus_vol': 'Sent + Vol',
    'D_with_interaction': 'All + Interaction',
}
results['Feature Set'] = results['feature_set'].map(label_map)

# ============================================================
# TABLE 1: Overall Results (3 × 4)
# ============================================================
print("=" * 80)
print("TABLE: Prediction Results — Overall (3 Models × 4 Feature Sets)")
print("=" * 80)

table1 = results[['model', 'Feature Set', 'overall_Accuracy', 'overall_F1', 
                   'overall_MCC', 'overall_AUC-ROC']].copy()
table1.columns = ['Model', 'Feature Set', 'Accuracy', 'F1', 'MCC', 'AUC-ROC']
table1[['Accuracy', 'F1', 'MCC', 'AUC-ROC']] = table1[['Accuracy', 'F1', 'MCC', 'AUC-ROC']].round(4)

print(table1.to_string(index=False))

table1.to_csv(os.path.join(OUTPUT_DIR, "Table_Prediction_Overall_3x4.csv"), index=False)
print("\nSaved: Table_Prediction_Overall_3x4.csv")

# Baseline
baseline = 0.5141
print(f"\nBaseline (majority class): {baseline}")

# ============================================================
# TABLE 2: Volatile vs Calm — Feature Set D only
# ============================================================
print("\n" + "=" * 80)
print("TABLE: Prediction Results — Feature Set D, Volatile vs Calm")
print("=" * 80)

fs_d = results[results['feature_set'] == 'D_with_interaction'].copy()
table2 = fs_d[['model', 'overall_Accuracy', 'volatile_Accuracy', 'calm_Accuracy',
               'overall_MCC', 'volatile_MCC', 'calm_MCC']].copy()
table2.columns = ['Model', 'Overall Acc', 'Volatile Acc', 'Calm Acc', 
                   'Overall MCC', 'Volatile MCC', 'Calm MCC']
table2[['Overall Acc', 'Volatile Acc', 'Calm Acc', 'Overall MCC', 'Volatile MCC', 'Calm MCC']] = \
    table2[['Overall Acc', 'Volatile Acc', 'Calm Acc', 'Overall MCC', 'Volatile MCC', 'Calm MCC']].round(4)

print(table2.to_string(index=False))

table2.to_csv(os.path.join(OUTPUT_DIR, "Table_Prediction_Volatile_vs_Calm.csv"), index=False)
print("\nSaved: Table_Prediction_Volatile_vs_Calm.csv")

TABLE: Prediction Results — Overall (3 Models × 4 Feature Sets)
              Model       Feature Set  Accuracy     F1     MCC  AUC-ROC
Logistic Regression    Sentiment only    0.4993 0.4787  0.0016   0.5005
Logistic Regression   Volatility only    0.4901 0.4361 -0.0140   0.4873
Logistic Regression        Sent + Vol    0.4972 0.4908 -0.0042   0.4968
Logistic Regression All + Interaction    0.4969 0.4926 -0.0049   0.4962
XGBoost (per-stock)    Sentiment only    0.4998 0.5123 -0.0010   0.4992
XGBoost (per-stock)   Volatility only    0.5001 0.5043  0.0006   0.4999
XGBoost (per-stock)        Sent + Vol    0.4996 0.5089 -0.0010   0.4998
XGBoost (per-stock) All + Interaction    0.4997 0.5090 -0.0010   0.4994
      LSTM (pooled)    Sentiment only    0.5042 0.5554  0.0029   0.5003
      LSTM (pooled)   Volatility only    0.4932 0.5087 -0.0146   0.4910
      LSTM (pooled)        Sent + Vol    0.5030 0.5617 -0.0008   0.4982
      LSTM (pooled) All + Interaction    0.4980 0.5467 -0.0094   0.4932


In [4]:
import pandas as pd
import os

OUTPUT_DIR = r"..\..\Dissertation_Results\Prediction"
decile = pd.read_csv(os.path.join(OUTPUT_DIR, "volatility_decile_results.csv"))

decile[['accuracy', 'f1', 'mcc', 'auc_roc']] = decile[['accuracy', 'f1', 'mcc', 'auc_roc']].round(4)

decile.columns = ['Decile', 'N', 'Accuracy', 'F1', 'MCC', 'AUC-ROC', 'Actual Up %']
decile['Actual Up %'] = (decile['Actual Up %'] * 100).round(1)

print("TABLE: Prediction Accuracy by Volatility Decile")
print(decile.to_string(index=False))

decile.to_csv(os.path.join(OUTPUT_DIR, "Table_Volatility_Decile_4dp.csv"), index=False)
print("\nSaved: Table_Volatility_Decile_4dp.csv")

TABLE: Prediction Accuracy by Volatility Decile
Decile     N  Accuracy     F1     MCC  AUC-ROC  Actual Up %
    D1 44320    0.4939 0.4786 -0.0101   0.4929         51.3
    D2 44918    0.4927 0.4851 -0.0133   0.4902         51.3
    D3 44984    0.4916 0.4924 -0.0153   0.4909         52.0
    D4 44075    0.4925 0.4938 -0.0142   0.4918         51.6
    D5 42866    0.4979 0.5017 -0.0037   0.4983         51.5
    D6 42511    0.4966 0.5047 -0.0067   0.4989         51.7
    D7 41931    0.4935 0.4780 -0.0120   0.4943         50.7
    D8 40819    0.5085 0.5147  0.0170   0.5082         51.0
    D9 38942    0.5097 0.5126  0.0197   0.5068         51.2
   D10 28387    0.4931 0.4488 -0.0061   0.4926         52.0

Saved: Table_Volatility_Decile_4dp.csv
